read the data.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_car = spark.read.format("parquet").option("header","true").option("inferSchema","true")\
    .load("abfss://bronze@pmngradls.dfs.core.windows.net/Raw Data/")

In [0]:
display(df_car)

In [0]:
df_car= df_car.withColumn("Model_category", split(col("Model_ID"), "-")[0])

In [0]:
# Revenue per Unit
df_car = df_car.withColumn("Revenue_Per_Unit", col("Revenue") / col("Units_Sold"))

# Revenue with Tax (e.g., 18% GST)
df_car = df_car.withColumn("Revenue_With_Tax", col("Revenue") * 1.18)

# Discounted Revenue
df_car = df_car.withColumn("Discounted_Revenue", col("Revenue") - (col("Revenue") * 0.10))

In [0]:
df_car.display()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
window = Window.partitionBy("Branch_ID").orderBy("Date_ID")
df_car.withColumn("Running_Revenue", sum("Revenue").over(window)).display()

#Rank Dealers by Revenue
window2 = Window.partitionBy("Branch_ID").orderBy(col("Revenue").desc())
df_car.withColumn("Dealer_Rank", rank().over(window2)).display()

AD HOC analysis

In [0]:
df_car.groupBy("year").agg(sum("Revenue").alias("total_revenue")).orderBy("year").display()

In [0]:
df_car.groupBy("year").agg(sum("Units_Sold").alias("total_units_sold")).orderBy("year").display()

In [0]:
#method 1

dfd = df_car.groupBy("branchName","year").agg(sum("Units_Sold").alias("total_units_sold"))\
        .orderBy(col("year"),col("total_units_sold").desc())

display(dfd)
window = Window.partitionBy("year").orderBy(col("total_units_sold").desc())
dfd.withColumn("dense_rank",dense_rank().over(window))


Databricks visualization. Run in Databricks to view.

In [0]:
dfd.withColumn("rank",rank().over(window)).display()

In [0]:
dfd.withColumn("row_number",row_number().over(window)).display()

In [0]:
df_car.write.format("parquet").mode("overwrite").save("abfss://silver@pmngradls.dfs.core.windows.net/Clean Data/")

read file using sql

In [0]:
%sql
select * from parquet.`abfss://silver@pmngradls.dfs.core.windows.net/Clean Data/`;